# Using ProbPipe to Create an Uncertainty-aware Forecasting Pipeline

In [ ]:
from probpipe.core.node import wf
from probpipe.core.node import abstractwf, WorkflowNode, ModuleNode, AbstractModule
from probpipe import MvNormal
from probpipe import EmpiricalDistribution, Distribution
import numpy as np
from numpy.typing import NDArray

from abc import ABC, abstractmethod
from typing import Type, TypeVar, Callable, Any


## Workflow Components

In [ ]:
# XXX: type hints in these node definitions will need to be updated to use the new typing definitions 
# probably want to have this in typing module 

TDistribution = TypeVar("TDistribution", bound=Distribution)


#####################################################################
### Likelihood modules can support computation of log-likelihoods ### 
### and/or generation of synthetic data                           ###
#####################################################################

class Likelihood(AbstractModule):
    """Abstract module node for (1) computing the likelihood of a model given data and parameters
    and (2) generating synthetic data given parameters.
    """
    # XXX: do abstactmethod and wf play nicely together?
    @abstractwf #it is like combination of @wf and @abstractmethod
    def log_likelihood(self,  params: NDArray, data: NDArray) -> float:
        pass


class GenerativeLikelihood(AbstractModule):
    @abstractwf
    def generate_data(self, params: NDArray, n_samples: int) -> NDArray:
        pass



class SimpleLikelihood(ModuleNode, Likelihood, GenerativeLikelihood):
    """A simple likelihood module that wraps a Distribution class."""
    def __init__(self, dist_cls: Type[TDistribution], params_name: str, **other_params):
        super().__init__()
        self.dist_cls = dist_cls
        # XXX: need to validate params_name and other_params match with dist_cls requirements
        self.params_name = params_name
        self.other_params = other_params

    def _get_distribution_for_params(self, params: NDArray) -> Distribution:
        dist_params = dict(self.other_params)
        dist_params[self.params_name] = params
        return self.dist_cls(**dist_params)
    @wf
    def log_likelihood(self, params: NDArray, data: NDArray) -> float:
        dist = self._get_distribution_for_params(params)
        return dist.log_density(data).sum()
    
    @wf
    def generate_data(self, params: NDArray, n_samples: int) -> NDArray:
        dist = self._get_distribution_for_params(params)
        return dist.sample(n_samples=n_samples)

#################################################################
### Creating and updating approximate posterior distributions ###
#################################################################

# XXX: it's awkward to have to specify T here; is there any alternative way to do this?
T = TypeVar("T", bound=np.number)
### Wrapper around Distribution object that tracks data, prior, and likelihood
class PosteriorDistribution(Distribution[T]):
    def __init__(self, posterior: Distribution[T], prior: Distribution[T], likelihood: Likelihood, data: NDArray):
        # XXX: is this the right way to set class so it behaves like a the Distribution type of posterior?
        self.__class__.__name__ = posterior.__class__.__name__
        self.posterior = posterior
        self.prior = prior
        self.likelihood = likelihood
        self.data = data

    def log_density(self, data: NDArray) -> NDArray[np.floating]:
        return self.prior.log_density(data) + self.likelihood.log_likelihood(params=data, data=self.data)
    
    def density(self, data: NDArray) -> NDArray[np.floating]:
        return np.exp(self.log_density(data))
    
    def sample(self, n_samples: int) -> NDArray:
        return self.posterior.sample(n_samples=n_samples)
    
    def predictive_density(self, data: NDArray) -> NDArray[np.floating]:
        return self.likelihood.log_likelihood(params=data, data=self.data)
    
    def sample_predictive(self, n_samples: int) -> NDArray:
        # XXX: actually should generate n_samples from posterior, then generate one sample from likelihood for each posterior sample
        return self.likelihood.generate_data(params=self.posterior.sample(n_samples=1), n_samples=n_samples)

    def expectation(self, func: Callable[[NDArray[T]], NDArray]) -> Distribution[T]:
        return self.posterior.expectation(func)
    
    @classmethod
    def from_distribution(
        cls,
        convert_from: Distribution, 
        **fit_kwargs: Any,
    ) -> Distribution[T]:
        raise NotImplementedError("from_distribution method not implemented for PosteriorDistribution")


class ApproximatePosterior(WorkflowNode, ABC):
    """Computes the posterior distribution of a model given prior, likelihood model, and data."""
    def __init__(self):
        super().__init__(
            func=self._compute_posterior,
            child_nodes={}, # doesn't make sense to pass this in to a workflow node; should be inferred from func signature
            inputs={}, # same here 
            prefect_kind='task',
            name='compute_posterior',
        )

    @abstractmethod
    def _compute_posterior(self, prior: Distribution, likelihood: Likelihood, data: NDArray) -> PosteriorDistribution:
        pass


class RWMH(ApproximatePosterior):
    """Computes the posterior distribution using a Random Walk Metropolis-Hastings algorithm with a multivariate Gaussian proposal."""
    def __init__(self, step_size: float = 1):
        super().__init__()
        self.step_size = step_size

    def _compute_posterior(self, prior: Distribution, likelihood: Likelihood, data: NDArray) -> PosteriorDistribution:
        # XXX: implement RWMH algorithm here
        post_approx = EmpiricalDistribution(samples=np.array([]))  # XXX: placeholder
        return PosteriorDistribution(posterior=post_approx, prior=prior, likelihood=likelihood, data=data) 


class IterativeForecaster(ModuleNode):
    """Can iteratively update posterior given new data and generate predictions from posterior."""
    def __init__(self, prior: Distribution, likelihood: Likelihood):
        super().__init__(child_nodes=dict(likelihood=likelihood), inputs={})
        self._curr_posterior = PosteriorDistribution(posterior=prior, prior=prior, likelihood=likelihood, data=np.array([]))

    def current_posterior(self) -> PosteriorDistribution:
        return self._curr_posterior

    @wf
    def update(
        self,
        approx_post: ApproximatePosterior,
        likelihood: Likelihood,
        data: NDArray,
    ) -> Distribution:
        post_dist = approx_post(prior=self.curr_posterior, likelihood=likelihood, data=data)
        self._curr_posterior = post_dist
        return post_dist

    @wf
    def forecast(
        self, 
        n_samples: int) -> NDArray:
        return self.curr_posterior.sample_predictive(n_samples=n_samples)

#########################
### Predictive checks ###
#########################

class PredictiveChecker(ModuleNode):
    def __init__(self, statistic: Callable[[NDArray], float]):
        super().__init__(child_nodes={}, inputs={})
        self.statistic = statistic

    @abstractmethod
    @wf
    def predictive_p_value(
        self, 
        posterior: PosteriorDistribution,
    ) -> float:
        pass

class PosteriorPredictiveChecker(PredictiveChecker):
    @wf
    def predictive_p_value(self, posterior: PosteriorDistribution) -> float:
        obs_data = posterior.data
        obs_stat = self.statistic(obs_data)
        sim_stats = np.array([self.statistic(posterior.sample_predictive(n_samples=1)) for _ in range(1000)])
        return (sim_stats >= obs_stat).mean()

## Example Usage

In [3]:
# note that we could easily swap out prior, likelihood, or approx_posterior with different choices
prior = MvNormal(mean=np.array([0, 0]), cov=np.eye(2))
likelihood = SimpleLikelihood(dist_cls=MvNormal, params_name='mean', cov=np.eye(2))
approx_post = RWMH(step_size=0.1)
# need IterativeForecaster constructor to automatically  take approx_posterior as an argument
forecaster = IterativeForecaster(prior=prior, likelihood=likelihood, approx_post=approx_post)
ppc = PosteriorPredictiveChecker(statistic=lambda x: x.mean(axis=0))

forecasts = []
ppc_pvalues = []
for i in range(10):
    # XXX: stand-in data generation for illustration purposes
    obs_data = np.random.multivariate_normal(mean=np.array([5, -3]), cov=4*np.eye(2), size=100)
    posterior_dist = forecaster.update(data=obs_data)
    forecast = forecaster.forecast(n_samples=100)
    forecasts.append(forecast)
    ppc_pvalues.append(ppc.predictive_p_value(posterior=posterior_dist))
    print(f"Posterior distribution after dataset {i}: {posterior_dist}")
    print(f"Posterior predictive p-value for dataset {i}: {ppc_pvalues[-1]}")

TypeError: Workflow 'compute_posterior' received unknown inputs {'prefect_kind'}